# Preprocessing of the Data. Download, import and get a first overview at the Data.

### Load packages

In [1]:
# Import required packages
import os
import sys
import time
import json
import numpy as np
import pandas as pd
import glob
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import sklearn
from dotenv import load_dotenv
from pathlib import Path

### Step: Join solar potential with roof segments
These cells load solar potential data and roof-segment slope data, then perform a spatial join between both layers.

In [2]:
# Load solar potential data (already clipped to Neukoelln)
project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

solar_potential_file_path = project_root / "data" / "exports" / "preprocessing_step1_clip_to_neuk" / "Solarpotential_Neukoelln.shp"
solar_potential_data = gpd.read_file(solar_potential_file_path)

# Print source file
print(f"Solar file: {solar_potential_file_path}")

Solar file: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step1_clip_to_neuk\Solarpotential_Neukoelln.shp


In [3]:
# Inspect attributes and keep full feature set
solar_potential_attribute_table = pd.DataFrame(
    solar_potential_data.drop(columns="geometry", errors="ignore")
)
print(solar_potential_attribute_table.columns)

# Optional: define a compact core-column view for quick checks
candidate_cols = [
    "geometry",
    "gebaeudefu",
    "bauweise",
    "ist_denkma",
    "anzahl_unt",
    "anzahl_obe",
    "verschattu",
    "verschat_1",
    "verschat_2",
]
available_cols = [c for c in candidate_cols if c in solar_potential_data.columns]
solar_potential_core = solar_potential_data[available_cols].copy()

# IMPORTANT: do not overwrite solar_potential_data; keep all columns for downstream exports
print("Core columns preview:", list(solar_potential_core.columns))
print("Full column count kept for pipeline:", len(solar_potential_data.columns))

Index(['gml_id', 'solarrechn', 'geeignete_', 'anzahl_sta', 'potenziell',
       'verschattu', 'verschat_1', 'verschat_2', 'gebaeudefu', 'lage',
       'hausnummer', 'bauweise', 'ist_denkma', 'denkmaltyp', 'gebaeudebe',
       'anzahl_unt', 'anzahl_obe', 'ist_hochha', 'strasse_sc', 'gebaeude_1',
       'bauweise_s', 'gebaeudepu', 'gebaeude_2', 'id'],
      dtype='str')
Core columns preview: ['geometry', 'gebaeudefu', 'bauweise', 'ist_denkma', 'anzahl_unt', 'anzahl_obe', 'verschattu', 'verschat_1', 'verschat_2']
Full column count kept for pipeline: 25


In [4]:
# Load green roofs and add solar attributes 
project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

# Load green roofs from Step 1 export
green_roofs_path = project_root / "data" / "exports" / "preprocessing_step1_clip_to_neuk" / "neukoelln_greenroofs.shp"
green_roofs = gpd.read_file(green_roofs_path)

# Build base roof layer with target column
if "target_int" in green_roofs.columns:
    roof_base = green_roofs[["geometry", "target_int"]].copy()
elif "target_0_1" in green_roofs.columns:
    roof_base = green_roofs[["geometry", "target_0_1"]].copy()
    roof_base = roof_base.rename(columns={"target_0_1": "target_int"})
elif "ex_int" in green_roofs.columns:
    roof_base = green_roofs[["geometry", "ex_int"]].copy()
    roof_base["target_int"] = roof_base["ex_int"].map({"extensiv": 1, "intensiv": 0, np.nan: 0})
    roof_base = roof_base.drop(columns=["ex_int"])
else:
    roof_base = green_roofs[["geometry"]].copy()
    roof_base["target_int"] = 0

# Keep roof area as reference cap for later suitability checks
if "roof_area_m2" in green_roofs.columns:
    roof_base["roof_area_m2"] = pd.to_numeric(green_roofs["roof_area_m2"], errors="coerce")
elif "roof_area_" in green_roofs.columns:
    roof_base["roof_area_m2"] = pd.to_numeric(green_roofs["roof_area_"], errors="coerce")

# Stable row id for robust joins
roof_base = roof_base.reset_index(drop=True).copy()
roof_base["roof_row_id"] = roof_base.index

# Continue directly with roof base 
green_roofs_with_floors = roof_base.copy()

# Add solar potential attributes using representative points to avoid intersects-duplicates
roof_points = green_roofs_with_floors[["roof_row_id", "geometry"]].copy()
roof_points = roof_points.set_geometry(roof_points.geometry.representative_point())

solar_join_points = gpd.sjoin(
    roof_points,
    solar_potential_data,
    how="left",
    predicate="within",
)
solar_join_points = solar_join_points.sort_values(["roof_row_id", "index_right"], na_position="last")
solar_join_points = solar_join_points.drop_duplicates(subset=["roof_row_id"], keep="first")

# Keep only solar attributes (without geometry/index helper columns), then merge back
solar_attr_cols = [
    c for c in solar_join_points.columns
    if c not in {"geometry", "index_right"}
    and c not in green_roofs_with_floors.columns
]
solar_attrs = solar_join_points[["roof_row_id"] + solar_attr_cols].copy()
green_roofs_with_solar = green_roofs_with_floors.merge(solar_attrs, on="roof_row_id", how="left")

# Stable building key from geometry to avoid double-counting segments later
green_roofs_with_solar["building_key"] = green_roofs_with_solar.geometry.apply(
    lambda g: g.wkb_hex if g is not None else None
)

# Load roof-segment slope data
neigung_file_path = (
    project_root
    / "data"
    / "solarpotential_roofsegments_neukoelln"
    / "roof_segments_Neukoelln.shp"
)
dachneigung_data = gpd.read_file(neigung_file_path)[["geometry", "neigung"]].copy()

print(f"Green-roofs file: {green_roofs_path}")
print(f"Roof-segment file: {neigung_file_path}")
print(f"Roofs with solar features: {len(green_roofs_with_solar)}")
print(f"Unique buildings by geometry: {green_roofs_with_solar['building_key'].nunique()}")
print(dachneigung_data.columns)

Green-roofs file: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step1_clip_to_neuk\neukoelln_greenroofs.shp
Roof-segment file: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\solarpotential_roofsegments_neukoelln\roof_segments_Neukoelln.shp
Roofs with solar features: 53337
Unique buildings by geometry: 53337
Index(['geometry', 'neigung'], dtype='str')


## Join roof segments to buildings

Roof surfaces are split into multiple slope segments, so each segment is assigned to a building first.
Then we evaluate the flat-roof share per building.

In [5]:
# Spatially join roof segments with unique building geometries
green_roofs_unique = green_roofs_with_solar.drop_duplicates(subset=["building_key"]).copy()

solar_with_segments = gpd.sjoin(
    dachneigung_data,
    green_roofs_unique,
    how="left",
    predicate="within",
)

print(solar_with_segments.columns)
print(f"Roof segments: {len(dachneigung_data)}")
print(f"Unique buildings used in join: {len(green_roofs_unique)}")
print(f"Joined segments: {len(solar_with_segments)}")

Index(['geometry', 'neigung', 'index_right', 'target_int', 'roof_area_m2',
       'roof_row_id', 'gml_id', 'solarrechn', 'geeignete_', 'anzahl_sta',
       'potenziell', 'verschattu', 'verschat_1', 'verschat_2', 'gebaeudefu',
       'lage', 'hausnummer', 'bauweise', 'ist_denkma', 'denkmaltyp',
       'gebaeudebe', 'anzahl_unt', 'anzahl_obe', 'ist_hochha', 'strasse_sc',
       'gebaeude_1', 'bauweise_s', 'gebaeudepu', 'gebaeude_2', 'id',
       'building_key'],
      dtype='str')
Roof segments: 104033
Unique buildings used in join: 53337
Joined segments: 104033


In [6]:
print(solar_with_segments.head())

                                            geometry  neigung  index_right  \
0  POLYGON ((397371.131 5808654.889, 397374.479 5...     36.3          NaN   
1  POLYGON ((397365.012 5808643.271, 397361.669 5...     40.3          NaN   
2  POLYGON ((397955.006 5808413.273, 397953.892 5...      2.8       9664.0   
3  POLYGON ((397656.094 5808222.917, 397652.934 5...      6.9       9341.0   
4  POLYGON ((397319.782 5808369.571, 397316.761 5...     65.7          NaN   

   target_int  roof_area_m2  roof_row_id             gml_id  \
0         NaN           NaN          NaN                NaN   
1         NaN           NaN          NaN                NaN   
2         0.0         67.72       9664.0  c_gebaeude.317544   
3         0.0         43.10       9341.0  c_gebaeude.322293   
4         NaN           NaN          NaN                NaN   

                                          solarrechn  geeignete_  anzahl_sta  \
0                                                NaN         NaN        

In [7]:
# Aggregate roof segments to building level and compute suitability share
SLOPE_THRESHOLD = 15.0  # Segments up to 15 degrees are treated as sufficiently flat

# Calculate areas in projected CRS (m2)
seg_calc = solar_with_segments.copy()
if seg_calc.crs is not None and seg_calc.crs.is_geographic:
    seg_calc = seg_calc.to_crs(25833)

seg_calc["segment_area"] = seg_calc.geometry.area

# Suitable area per segment (only where slope <= threshold)
seg_calc["suitable_area"] = np.where(
    seg_calc["neigung"] <= SLOPE_THRESHOLD,
    seg_calc["segment_area"],
    0.0,
)

# Aggregate to building level via stable key
building_df = seg_calc.groupby("building_key", dropna=True).agg({
    "suitable_area": "sum",
    "segment_area": "sum",
}).reset_index()

# Add roof reference area and cap impossible values
roof_area_ref = (
    green_roofs_unique[["building_key", "roof_area_m2"]]
    .dropna(subset=["building_key"])
    .drop_duplicates(subset=["building_key"])
    .copy()
)
building_df = building_df.merge(roof_area_ref, on="building_key", how="left")

# Use roof area as physical cap when available
has_roof_area = building_df["roof_area_m2"].notna()
building_df.loc[has_roof_area, "segment_area"] = np.minimum(
    building_df.loc[has_roof_area, "segment_area"],
    building_df.loc[has_roof_area, "roof_area_m2"],
)
building_df.loc[has_roof_area, "suitable_area"] = np.minimum(
    building_df.loc[has_roof_area, "suitable_area"],
    building_df.loc[has_roof_area, "roof_area_m2"],
)

# Safety: suitable area can never exceed total segment area
building_df["suitable_area"] = np.minimum(
    building_df["suitable_area"],
    building_df["segment_area"],
)

# Compute suitability percentage (prefer roof_area_m2 denominator if available)
denominator = np.where(
    has_roof_area,
    building_df["roof_area_m2"],
    building_df["segment_area"],
)
building_df["suitability_pct"] = np.where(
    denominator > 0,
    (building_df["suitable_area"] / denominator) * 100,
    np.nan,
)

# Export suitability table
project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent
output_dir = project_root / "data" / "exports" / "preprocessing_step2_sol_suit"
output_dir.mkdir(parents=True, exist_ok=True)

building_csv_path = output_dir / "solar_segment_suitability.csv"
building_df.to_csv(building_csv_path, index=False)

print(f"Suitability table saved: {building_csv_path}")
print(f"Number of buildings: {len(building_df)}")
print(f"Max suitability_pct: {building_df['suitability_pct'].max():.2f}")
print(f"Rows with suitable_area > roof_area_m2: {((building_df['suitable_area'] > building_df['roof_area_m2']) & building_df['roof_area_m2'].notna()).sum()}")

Suitability table saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step2_sol_suit\solar_segment_suitability.csv
Number of buildings: 28491
Max suitability_pct: 100.00
Rows with suitable_area > roof_area_m2: 0


In [8]:
# Attach suitability metrics back to enriched roof-building data
final_buildings = green_roofs_with_solar.merge(
    building_df.drop(columns=["segment_area"], errors="ignore"),
    on="building_key",
    how="left",
)

In [9]:
print(final_buildings.columns)
print(final_buildings.head())

Index(['geometry', 'target_int', 'roof_area_m2_x', 'roof_row_id', 'gml_id',
       'solarrechn', 'geeignete_', 'anzahl_sta', 'potenziell', 'verschattu',
       'verschat_1', 'verschat_2', 'gebaeudefu', 'lage', 'hausnummer',
       'bauweise', 'ist_denkma', 'denkmaltyp', 'gebaeudebe', 'anzahl_unt',
       'anzahl_obe', 'ist_hochha', 'strasse_sc', 'gebaeude_1', 'bauweise_s',
       'gebaeudepu', 'gebaeude_2', 'id', 'building_key', 'suitable_area',
       'roof_area_m2_y', 'suitability_pct'],
      dtype='str')
                                            geometry  target_int  \
0  POLYGON ((397257.326 5806354.677, 397258.326 5...           0   
1  POLYGON ((397282.326 5806352.677, 397283.326 5...           0   
2  POLYGON ((397271.326 5806370.677, 397272.326 5...           0   
3  POLYGON ((397313.435 5806380.608, 397314.76 58...           0   
4  POLYGON ((397265.326 5806378.677, 397266.326 5...           0   

   roof_area_m2_x  roof_row_id             gml_id  \
0           59.00       

In [10]:
# Configure export paths and prepare the GeoDataFrame
project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent
output_dir = project_root / "data" / "exports" / "preprocessing_step2_sol_suit"
output_dir.mkdir(parents=True, exist_ok=True)

final_csv_path = output_dir / "preprocessed_buildings_sol_suit.csv"
final_gpkg_path = output_dir / "preprocessed_buildings_sol_suit.gpkg"
final_shp_path = output_dir / "preprocessed_buildings_sol_suit.shp"

export_gdf = final_buildings.copy()

# Keep Step 2 output compatible with the previously used Step 3 schema
if "index_right" in export_gdf.columns and "index_righ" not in export_gdf.columns:
    export_gdf = export_gdf.rename(columns={"index_right": "index_righ"})

if "index" not in export_gdf.columns:
    if "index_righ" in export_gdf.columns:
        export_gdf["index"] = export_gdf["index_righ"]
    else:
        export_gdf["index"] = export_gdf.index

if "ex_int" not in export_gdf.columns and "target_int" in export_gdf.columns:
    export_gdf["ex_int"] = export_gdf["target_int"]

if "suitable_a" not in export_gdf.columns and "suitable_area" in export_gdf.columns:
    export_gdf["suitable_a"] = export_gdf["suitable_area"]
if "segment_ar" not in export_gdf.columns and "segment_area" in export_gdf.columns:
    export_gdf["segment_ar"] = export_gdf["segment_area"]
if "suitabilit" not in export_gdf.columns and "suitability_pct" in export_gdf.columns:
    export_gdf["suitabilit"] = export_gdf["suitability_pct"]

legacy_columns = [
    "index",
    "ex_int",
    "gebaeudefu",
    "bauweise",
    "ist_denkma",
    "anzahl_unt",
    "anzahl_obe",
    "verschattu",
    "verschat_1",
    "verschat_2",
    "index_righ",
    "suitable_a",
    "segment_ar",
    "suitabilit",
]
keep_columns = [c for c in legacy_columns if c in export_gdf.columns]
if "geometry" in export_gdf.columns:
    keep_columns.append("geometry")
export_gdf = export_gdf[keep_columns].copy()

print("Step 2 export columns:", keep_columns)

# Rename duplicate columns safely (common to_file failure cause)
if export_gdf.columns.duplicated().any():
    seen = {}
    new_cols = []
    for col in export_gdf.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    export_gdf.columns = new_cols

# Convert problematic object values (lists/dicts) to strings
for col in export_gdf.columns:
    if col == "geometry":
        continue
    if export_gdf[col].dtype == "object":
        export_gdf[col] = export_gdf[col].apply(
            lambda v: v if isinstance(v, (str, int, float, bool, type(None))) else str(v)
        )

Step 2 export columns: ['index', 'ex_int', 'gebaeudefu', 'bauweise', 'ist_denkma', 'anzahl_unt', 'anzahl_obe', 'verschattu', 'verschat_1', 'verschat_2', 'suitable_a', 'suitabilit', 'geometry']


#### Export cleaned outputs
This cell writes CSV, GPKG, and SHP outputs from the prepared final GeoDataFrame.

In [11]:
# Export CSV without geometry
export_gdf.drop(columns=["geometry"], errors="ignore").to_csv(final_csv_path, index=False)

# Export full layer to GPKG
export_gdf.to_file(final_gpkg_path, layer="preprocessed_buildings", driver="GPKG", index=False)

# Export SHP only for Polygon/MultiPolygon geometries
polygon_types = {"Polygon", "MultiPolygon"}
final_buildings_poly = export_gdf[export_gdf.geometry.geom_type.isin(polygon_types)].copy()
if len(final_buildings_poly) > 0:
    try:
        final_buildings_poly.to_file(final_shp_path, index=False)
        print(f"SHP saved: {final_shp_path} ({len(final_buildings_poly)} features)")
    except PermissionError:
        fallback_shp_path = final_shp_path.with_name("preprocessed_buildings_latest.shp")
        final_buildings_poly.to_file(fallback_shp_path, index=False)
        print(
            "Original SHP is locked by another process. "
            f"Saved fallback SHP: {fallback_shp_path} ({len(final_buildings_poly)} features)"
        )
else:
    print("No SHP exported: no polygon geometries found.")

print(f"CSV saved: {final_csv_path}")
print(f"GPKG saved: {final_gpkg_path} ({len(export_gdf)} features)")

SHP saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step2_sol_suit\preprocessed_buildings_sol_suit.shp (53337 features)
CSV saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step2_sol_suit\preprocessed_buildings_sol_suit.csv
GPKG saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step2_sol_suit\preprocessed_buildings_sol_suit.gpkg (53337 features)


In [12]:
# Step 2 inputs are already clipped to Neukölln in preprocessing_step1
# No additional district clipping is needed here.
project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

export_dir = project_root / "data" / "exports" / "preprocessing_step2_sol_suit"
preprocessed_candidates = [
    export_dir / "preprocessed_buildings_sol_suit.gpkg",
    export_dir / "preprocessed_buildings_sol_suit.shp",
    export_dir / "preprocessed_buildings_sol_suit.shp",
]
preprocessed_path = next((p for p in preprocessed_candidates if p.exists()), None)
if preprocessed_path is None:
    raise FileNotFoundError(
        "No preprocessed file found. Expected: "
        + " or ".join(str(p) for p in preprocessed_candidates)
    )

print("District clip skipped: files are already Neukölln-clipped.")
print(f"Using preprocessed file: {preprocessed_path}")

District clip skipped: files are already Neukölln-clipped.
Using preprocessed file: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step2_sol_suit\preprocessed_buildings_sol_suit.gpkg


In [13]:
# Convert joined solar-segment data to a pandas attribute table
solar_with_segments_attribute_table = pd.DataFrame(
    solar_with_segments.drop(columns="geometry", errors="ignore")
)
print(solar_with_segments_attribute_table.head())

   neigung  index_right  target_int  roof_area_m2  roof_row_id  \
0     36.3          NaN         NaN           NaN          NaN   
1     40.3          NaN         NaN           NaN          NaN   
2      2.8       9664.0         0.0         67.72       9664.0   
3      6.9       9341.0         0.0         43.10       9341.0   
4     65.7          NaN         NaN           NaN          NaN   

              gml_id                                         solarrechn  \
0                NaN                                                NaN   
1                NaN                                                NaN   
2  c_gebaeude.317544  https://solarrechner.berlin.de/solarrechner?g=...   
3  c_gebaeude.322293  https://solarrechner.berlin.de/solarrechner?g=...   
4                NaN                                                NaN   

   geeignete_  anzahl_sta  potenziell  ...  anzahl_unt  anzahl_obe  \
0         NaN         NaN         NaN  ...         NaN         NaN   
1         Na